# Snippet editor (Jupyter)

Run the setup cell once, then **only edit** the next code cell:

- **`CATEGORY`** — Sidebar name. Use the **exact** name of an existing category (e.g. `Git`, `.NET CLI`) to **append** there, or a **new** name to create `data/<slug>.json`.
- **`NEW_SNIPPETS`** — List of dicts with only **`title`**, **`code`**, **`desc`**. Use triple quotes for multiline code: `code = """..."""`.

Everything else is automatic: **file name**, **icon**, **snippet ids**, **merge** → `snippets.json`.

Requires Python 3 stdlib only; merge calls `tools/merge_snippets.py`.

In [1]:
from __future__ import annotations

import json
import re
import subprocess
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
if (_cwd / "data").is_dir():
    REPO_ROOT = _cwd
elif (_cwd / "merge_snippets.py").is_file():
    REPO_ROOT = _cwd.parent
else:
    REPO_ROOT = _cwd

# REPO_ROOT = Path("/path/to/CodeSnippet").resolve()  # if auto-detect fails

DATA_DIR = REPO_ROOT / "data"
MERGE_SCRIPT = REPO_ROOT / "tools" / "merge_snippets.py"

assert DATA_DIR.is_dir(), (
    f"Missing data dir: {DATA_DIR}\n"
    "Open Jupyter with cwd = repo root (folder that contains data/)."
)


def slugify(name: str) -> str:
    s = name.lower().strip()
    s = re.sub(r"[^a-z0-9]+", "-", s)
    s = s.strip("-") or "category"
    return s


def guess_icon_key(category_name: str) -> str:
    n = category_name.lower()
    if "git" in n:
        return "GitBranch"
    if "sql" in n or "database" in n:
        return "Database"
    if any(x in n for x in ("docker", "bash", "shell", "terminal", "cli", "npm", "node")):
        return "Terminal"
    if "c#" in n or "csharp" in n:
        return "Hash"
    if ".net" in n or "dotnet" in n:
        return "Terminal"
    return "Code"


def list_data_files() -> list[Path]:
    return sorted(DATA_DIR.glob("*.json"))


def collect_all_ids() -> set[str]:
    ids: set[str] = set()
    for p in list_data_files():
        data = json.loads(p.read_text(encoding="utf-8"))
        for s in data.get("snippets", []):
            sid = s.get("id", "")
            if sid in ids:
                raise ValueError(f"Duplicate id across data files: {sid}")
            ids.add(sid)
    return ids


def find_category_file(display_name: str) -> tuple[Path | None, dict | None]:
    want = display_name.strip().lower()
    for p in list_data_files():
        d = json.loads(p.read_text(encoding="utf-8"))
        if str(d.get("category", "")).strip().lower() == want:
            return p, d
    return None, None


def infer_id_prefix(data: dict) -> str:
    """Use existing ids in file (e.g. net-1 → net); else slug from category."""
    for s in data.get("snippets") or []:
        sid = str(s.get("id", ""))
        m = re.match(r"^(.+)-(\d+)$", sid)
        if m:
            return m.group(1)
    return slugify(str(data.get("category", "snippet")))


def next_snippet_id(prefix: str, pool: set[str] | None = None) -> str:
    ids = set(pool) if pool is not None else collect_all_ids()
    pat = re.compile(rf"^{re.escape(prefix)}-(\d+)$")
    highest = 0
    for sid in ids:
        m = pat.match(str(sid))
        if m:
            highest = max(highest, int(m.group(1)))
    n = highest + 1
    cand = f"{prefix}-{n}"
    while cand in ids:
        n += 1
        cand = f"{prefix}-{n}"
    return cand


def ensure_snippet_ids(prefix: str, snippets: list[dict], pool: set[str] | None = None) -> list[dict]:
    used = set(pool) if pool is not None else collect_all_ids()
    out: list[dict] = []
    for raw in snippets:
        s = dict(raw)
        if not str(s.get("id", "")).strip():
            s["id"] = next_snippet_id(prefix, used)
            used.add(s["id"])
        out.append(s)
    return out


def save_category_file(filename: str, payload: dict, *, indent: int = 2) -> Path:
    if not filename.endswith(".json"):
        filename += ".json"
    path = DATA_DIR / filename
    path.write_text(json.dumps(payload, indent=indent, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"Wrote {path}")
    return path


def run_merge() -> None:
    if not MERGE_SCRIPT.is_file():
        print("Run from repo root: python3 tools/merge_snippets.py")
        return
    r = subprocess.run(
        [sys.executable, str(MERGE_SCRIPT)],
        cwd=REPO_ROOT,
        capture_output=True,
        text=True,
    )
    print(r.stdout or "", end="")
    if r.stderr:
        print(r.stderr, file=sys.stderr)
    r.check_returncode()
    print("Done. Commit data/*.json and snippets.json together.")


def add_snippets_automatic(category_display: str, items: list[dict], *, merge: bool = True) -> None:
    """Append to existing category (matched by name) or create a new JSON file; assign ids; optional merge."""
    if not str(category_display).strip():
        raise ValueError("CATEGORY cannot be empty")
    if not items:
        raise ValueError("NEW_SNIPPETS must contain at least one snippet")
    normalized: list[dict] = []
    for i, it in enumerate(items):
        for k in ("title", "code", "desc"):
            if k not in it:
                raise ValueError(f"Snippet {i}: missing '{k}'")
        normalized.append(
            {"title": str(it["title"]), "code": str(it["code"]), "desc": str(it["desc"])}
        )

    path, data = find_category_file(category_display)
    existing = collect_all_ids()

    if path is not None and data is not None:
        prefix = infer_id_prefix(data)
        filled = ensure_snippet_ids(prefix, normalized)
        for s in filled:
            if s["id"] in existing:
                raise ValueError(f"id already used: {s['id']}")
            data.setdefault("snippets", []).append(s)
            existing.add(s["id"])
        save_category_file(path.name, data)
        print(f"Appended {len(filled)} snippet(s) to existing category → {path.name}")
    else:
        slug = slugify(category_display)
        out_path = DATA_DIR / f"{slug}.json"
        if out_path.exists():
            raise FileExistsError(
                f"{out_path.name} exists but category label does not match {category_display!r}. "
                "Rename the file or match the JSON \"category\" field exactly."
            )
        prefix = slug
        filled = ensure_snippet_ids(prefix, normalized)
        for s in filled:
            if s["id"] in existing:
                raise ValueError(f"id already used: {s['id']}")
        payload = {
            "category": category_display.strip(),
            "iconKey": guess_icon_key(category_display),
            "snippets": filled,
        }
        save_category_file(f"{slug}.json", payload)
        print(f"Created category {category_display!r} → {slug}.json (icon: {payload['iconKey']})")

    if merge:
        run_merge()


print("REPO_ROOT:", REPO_ROOT)
print("Ready. Edit CATEGORY + NEW_SNIPPETS in the next cell, then run it.")

REPO_ROOT: /Users/codefrydev/Desktop/SourceCode/CodeSnippet
Ready. Edit CATEGORY + NEW_SNIPPETS in the next cell, then run it.


## Add snippets — **edit and run this cell only**

Use the **same** `CATEGORY` string as the sidebar for existing data (e.g. `C#`, `.NET CLI`). For a new group, type the new name.

In [2]:
CATEGORY = "Git"

NEW_SNIPPETS = [
    {
        "title": "Show remote URLs",
        "code": """git remote -v""",
        "desc": "Lists configured remotes with fetch and push URLs.",
    },
]

add_snippets_automatic(CATEGORY, NEW_SNIPPETS)

Wrote /Users/codefrydev/Desktop/SourceCode/CodeSnippet/data/git.json
Appended 1 snippet(s) to existing category → git.json
Wrote 4 categories to /Users/codefrydev/Desktop/SourceCode/CodeSnippet/snippets.json
Done. Commit data/*.json and snippets.json together.


## Regenerate `snippets.json` only

If you edited `data/*.json` by hand, run:

In [3]:
run_merge()

Wrote 4 categories to /Users/codefrydev/Desktop/SourceCode/CodeSnippet/snippets.json
Done. Commit data/*.json and snippets.json together.
